In [ ]:
!pip install google-generativeai pytest

In [ ]:
import os
import json
import re
import sys
import time
import pathlib
import datetime
import tempfile
import subprocess
import textwrap

from typing import Tuple


In [ ]:
import os
import re
import time
import tempfile
import subprocess
import textwrap
import json
import datetime
import pathlib
import sys
from typing import Tuple


In [ ]:
import os
print(os.environ.get("GEMINI_API_KEY"))


In [ ]:
from google.genai import Client
import os

API_KEY = os.environ["GEMINI_API_KEY"]

client = Client(api_key=API_KEY)

MODEL_NAME = "models/gemini-3.6-flash"

print("Gemini model loaded:", MODEL_NAME)


In [ ]:
# Light rate limit and retry caps
SLEEP_BETWEEN_CALLS_SEC = float(os.environ.get("RATE_SLEEP_SEC", "1.0"))
MAX_DEBUG_ROUNDS = int(os.environ.get("MAX_DEBUG_ROUNDS", "2"))

# Logging and run artifacts
RUN_DIR = pathlib.Path("runs") / datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_DIR / "log.jsonl"

def log_event(kind: str, payload: dict):
    rec = {
        "ts": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "kind": kind,
        **payload,
    }
    with LOG_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Helpers
CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)

def extract_code_block(text: str) -> str:
    m = CODE_BLOCK_RE.search(text or "")
    if m:
        return textwrap.dedent(m.group(1).strip())
    return textwrap.dedent(text or "").strip()

def run_py_tests(solution_code: str, test_code: str, timeout_sec: int = 10) -> Tuple[bool, str]:
    with tempfile.TemporaryDirectory() as tmp:
        sol = pathlib.Path(tmp) / "solution.py"
        tst = pathlib.Path(tmp) / "test_solution.py"
        sol.write_text(solution_code, encoding="utf-8")
        tst.write_text(test_code, encoding="utf-8")

        cmd = [sys.executable, str(tst)]
        try:
            proc = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=timeout_sec,
                cwd=tmp,
            )
            out = (proc.stdout or "") + (proc.stderr or "")
            passed = proc.returncode == 0
            return passed, out.strip()
        except subprocess.TimeoutExpired as e:
            return False, f"TIMEOUT after {timeout_sec}s\n{e}"
        except Exception as e:
            return False, f"EXEC_ERROR: {e}"


In [ ]:
# === Step 3.2: Define Agents ===

class GeminiAgent:
    def __init__(self, name: str, system_message: str):
        self.name = name
        self.system_message = system_message

    def ask(self, user_message: str, label: str = "") -> str:
        prompt = f"{self.system_message}\n\nUser: {user_message}"

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        return response.text or ""


# CodeGen agent
codegen = GeminiAgent(
    name="codegen",
    system_message=(
        "You are CodeGen. Your job is to write solution.py. "
        "Return ONLY a Python code block. No explanations."
    )
)

# Tester agent
tester = GeminiAgent(
    name="tester",
    system_message=(
        "You are Tester. You run tests on solution.py. "
        "Return ONLY a Python code block containing test_solution.py."
    )
)

# Debugger agent
debugger = GeminiAgent(
    name="debugger",
    system_message=(
        "You are Debugger. You analyze failing tests and rewrite solution.py. "
        "Return ONLY a Python code block."
    )
)


In [ ]:
class GeminiAgent:
    def __init__(self, name: str, system_message: str):
        self.name = name
        self.system_message = system_message

    def ask(self, user_message: str, label: str = "") -> str:
        prompt = f"{self.system_message}\n\nUser: {user_message}"

        response = client.responses.generate(
    model=MODEL_NAME,
    input=prompt
)


        # google-genai returns .text directly
        return response.text or ""


In [ ]:
def run_round(solution_code: str, test_code: str):
    passed, output = run_py_tests(solution_code, test_code)
    return passed, output


In [ ]:
cg_out = codegen.ask(
    user_message="Write solution.py as specified. Return ONLY the code block.",
    label="codegen"
)

solution = extract_code_block(cg_out)

log_event("solution_initial", {"code": solution})

print("=== CodeGen produced solution.py ===")
print(solution)


In [ ]:
client.models.list()

In [ ]:
for m in client.models.list():
    print(m.name)


In [ ]:
log_event("solution_initial", {"code": solution})


In [ ]:
MAX_DEBUG_ROUNDS = 5

def main():
    print("== Gemini Code Testing and Debugging Chain ==")
    print(f"Model: {MODEL_NAME}")
    log_event("start", {"model": MODEL_NAME})

    # 1) CodeGen creates solution.py
    cg_out = codegen.ask(
        user_message="Write solution.py as specified. Return ONLY the code block.",
        label="codegen"
    )
    solution = extract_code_block(cg_out)

    # Optional retry if CodeGen didn't follow instructions
    if "def solve" not in solution and "def is_palindrome" not in solution:
        cg_out = codegen.ask(
            user_message="Please include the required function signature.",
            label="codegen_retry"
        )
        solution = extract_code_block(cg_out)

    log_event("solution_initial", {"code": solution})
    print("\n[CodeGen] Generated solution.py")

    # 2) Tester creates test_solution.py
    tt_out = tester.ask(
        user_message="Create test_solution.py; ONLY code block.",
        label="tester"
    )
    tests = extract_code_block(tt_out)

    log_event("tests_initial", {"code": tests})
    print("[Tester] Generated test_solution.py")

    # 3) Execute tests
    passed, output = run_py_tests(solution, tests)
    print("\n[Test Runner] First run passed:", passed)
    print(output)

    log_event("test_run_initial", {"passed": passed, "output": output})

    # 4) Debugger loop
    rounds = 0
    while not passed and rounds < MAX_DEBUG_ROUNDS:
        rounds += 1
        print(f"\n[Debugger] Fix attempt #{rounds}")

        dbg_prompt = (
            "Tests failed with output below. Current solution.py is also provided.\n\n"
            f"=== FAIL LOG ===\n{output}\n\n"
            f"=== CURRENT solution.py ===\n```python\n{solution}\n```"
        )

        dbg_out = debugger.ask(
            user_message=dbg_prompt,
            label=f"debugger_round_{rounds}"
        )
        fixed = extract_code_block(dbg_out)

        if fixed and fixed != solution:
            solution = fixed
            log_event("debug_fix", {"round": rounds, "code": solution})

            passed, output = run_py_tests(solution, tests)
            print("[Test Runner] Passed after fix:", passed)
            print(output)

            log_event("test_rerun", {
                "round": rounds,
                "passed": passed,
                "output": output
            })

    # 5) Save artifacts
    (RUN_DIR / "solution.py").write_text(solution, encoding="utf-8")
    (RUN_DIR / "test_solution.py").write_text(tests, encoding="utf-8")

    status = "ALL TESTS PASSED" if passed else "STOPPED WITH FAILURES"
    print("\n== RESULT:", status)
    print("Artifacts saved to:", RUN_DIR)

    log_event("end", {"status": status, "artifacts_dir": str(RUN_DIR)})


# Run the orchestrator
main()
